# DỰ ÁN 1: 🌍PHÂN TÍCH VÀ DỰ BÁO NHIỆT ĐỘ TOÀN CẦU
## Notebook 07: AI DEPLOYMENT VỚI FASTAPI VÀ STREAMLIT

### 1. Mục tiêu
Triển khai mô hình Machine Learning dự báo nhiệt độ (đã huấn luyện ở Notebook 06) thành một ứng dụng thực tế. Người dùng có thể dự báo thông qua giao diện Web (Streamlit) và hệ thống backend xử lý API (FastAPI).

### 2. Kiến trúc hệ thống
```mermaid
graph TD;
    User-->|Tương tác|Streamlit_Dashboard;
    Streamlit_Dashboard-->|REST API Request|FastAPI_Backend;
    FastAPI_Backend-->|Load Model|XGBoost_Model;
    FastAPI_Backend-->|Truy vấn đặc trưng|PostgreSQL_Database;
```

Mối liên hệ:
- **Notebook 06 (Machine Learning)** -> **Notebook 07 (AI Deployment)** -> **Người sử dụng**

### 3. Xây dựng Backend với FastAPI (`api.py`)
Chúng ta sẽ sử dụng magic command `%%writefile` để tạo trực tiếp file `api.py`. File này sẽ chứa toàn bộ code Backend định nghĩa các endpoint và xử lý gọi mô hình dự báo.

In [7]:
%%writefile api.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import pandas as pd
import joblib
import uvicorn

# == 1. KHỞI TẠO FASTAPI ===
app = FastAPI(
    title="Climate Change Temperature Prediction API",
    description="API dự đoán nhiệt độ trung bình hàng tháng dựa trên mô hình XGBoost.",
    version="1.0"
)

# == 2. LOAD MODEL ===
# Giả sử bạn đã lưu mô hình tốt nhất từ Notebook 06 vào thư mục models/ (hoặc từ Model.py vào thư mục model/)
try:
    # Load file pkl mô hình ở đây
    model = joblib.load("../model/xgboost_model.pkl") 
    print("Đã load mô hình XGBoost thành công")
except Exception as e:
    print("Cảnh báo load model:", e)
    model = None

# == 3. MÔ TẢ CẤU TRÚC DỮ LIỆU ===
class PredictionInput(BaseModel):
    year: int
    month: int
    lag_1: float
    lag_12: float
    rolling_mean_12: float

# == 4. CÁC ENDPOINT API ===
@app.post("/predict")
def predict_temperature(data: PredictionInput):
    if model is None:
        raise HTTPException(status_code=500, detail="Model chưa được nạp. Hãy huấn luyện và lưu file mô hình trước.")
    
    input_df = pd.DataFrame([data.dict()])
    
    try:
        pred = model.predict(input_df)[0]
        return {
            "status": "success",
            "predicted_temperature": float(pred)
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# == 5. CHẠY APP ===
if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)


Overwriting api.py


### 4. Xây dựng Dashboard Frontend với Streamlit (`streamlit_app.py`)
Tiếp theo, tạo file `streamlit_app.py` chứa giao diện tương tác cho người dùng. Dashboard này sẽ lấy thông số đầu vào từ người dùng và gọi tới Backend FastAPI để lấy kết quả dự báo.

In [8]:
%%writefile streamlit_app.py
import streamlit as st
import requests

st.set_page_config(page_title="Dự Báo Nhiệt Độ", page_icon="🌍", layout="wide")

st.title("🌍 Dashboard Dự Báo Nhiệt Độ Bề Mặt Trái Đất")
st.markdown("Ứng dụng dự báo nhiệt độ trung bình hàng tháng dựa trên dữ liệu lịch sử và mô hình Machine Learning.")

# Form nhập liệu ở Sidebar
st.sidebar.header("Thông số đầu vào")
year = st.sidebar.number_input("Năm (Year)", min_value=2000, max_value=2100, value=2026)
month = st.sidebar.slider("Tháng (Month)", 1, 12, 8)
lag_1 = st.sidebar.number_input("Nhiệt độ tháng trước (Lag 1) °C", value=25.0)
lag_12 = st.sidebar.number_input("Nhiệt độ cùng kỳ năm ngoái (Lag 12) °C", value=24.5)
rolling_mean_12 = st.sidebar.number_input("Nhiệt độ trung bình 12 tháng (Rolling Mean) °C", value=24.8)

if st.sidebar.button("🚀 Dự Báo Ngay"):
    # URL của FastAPI Backend
    api_url = "http://localhost:8000/predict"
    payload = {
        "year": year,
        "month": month,
        "lag_1": lag_1,
        "lag_12": lag_12,
        "rolling_mean_12": rolling_mean_12
    }
    
    with st.spinner("Đang tính toán dự báo..."):
        try:
            response = requests.post(api_url, json=payload)
            if response.status_code == 200:
                result = response.json()
                pred_temp = result["predicted_temperature"]
                st.success("✅ Hoàn tất!")
                
                # Hiển thị Dashboard trực quan (KPI)
                col1, col2, col3 = st.columns(3)
                col1.metric("Nhiệt độ tháng trước", f"{lag_1:.2f} °C")
                col2.metric("Nhiệt độ dự báo", f"{pred_temp:.2f} °C", f"{pred_temp - lag_1:.2f} °C so với tháng trước")
                col3.metric("Nhiệt độ TB 12 tháng", f"{rolling_mean_12:.2f} °C")
                
            else:
                st.error(f"Lỗi từ API Backend: Mã lỗi {response.status_code}")
        except requests.exceptions.ConnectionError:
            st.error("❌ Không thể kết nối tới Backend. Hãy chắc chắn rằng bạn đã chạy FastAPI (uvicorn api:app).")


Overwriting streamlit_app.py


### 5. Hướng dẫn chạy ứng dụng (Deployment Instructions)

Để chạy ứng dụng hoàn chỉnh trên máy cục bộ, bạn cần mở **2 Terminal (Command Prompt / PowerShell)** riêng biệt và trỏ thư mục về nơi lưu file `api.py` và `streamlit_app.py`:

**Terminal 1 - Chạy FastAPI Backend:**
```bash
# Cài đặt nhanh thư viện nếu chưa có: pip install fastapi uvicorn streamlit requests joblib pydantic

# Khởi động FastAPI với uvicorn
uvicorn api:app --reload
```
*Sau khi chạy, Backend sẽ hoạt động tại: `http://localhost:8000` và Swagger UI (tài liệu test API) tại `http://localhost:8000/docs`*

---

**Terminal 2 - Chạy Streamlit Frontend:**
```bash
# Khởi động giao diện Streamlit Dashboard
streamlit run streamlit_app.py
```
*Trình duyệt sẽ tự động mở giao diện ứng dụng web của bạn tại `http://localhost:8501`.*

### 6. Kết luận
Qua 7 notebook, chúng ta đã hoàn thành vòng đời của dự án Machine Learning từ bước khám phá dữ liệu (Notebook 01), xử lý làm sạch (Notebook 03), trích xuất đặc trưng (Notebook 05), huấn luyện đánh giá mô hình (Notebook 06) đến triển khai sản phẩm cuối (Notebook 07) để người dùng có thể thao tác dự báo trực quan thông qua Web Dashboard Streamlit và REST API Backend.